# Scout Trino Delta Lake Quickstart

Notebooks query the Delta Lake through **Trino**. Every read goes through Trino's HTTPS endpoint, which authenticates you via your Keycloak access token and applies per-user row filtering / column masking via OPA based on your Keycloak attributes (`allowed_facilities`, `allowed_modalities`, `mask_phi_fields`).

The `scout_trino.query(sql)` helper handles authentication transparently — access tokens are refreshed via the JupyterHub API as needed, so sessions stay valid for as long as your Keycloak SSO session is active (default ~30 days idle). No re-login required mid-session, no `KEYCLOAK_ACCESS_TOKEN` env var, no manual refresh.

In [ ]:
import os
import pandas as pd
from scout_trino import query

In [ ]:
# Count the reports in the delta lake.
# `reports` contains all report versions; `reports_latest` is deduplicated to
# the most recent version per report. Most analyses should use
# `reports_latest`. Note: some studies are read in parts and may be split
# into distinct reports; `reports_latest` will not capture those.
query("SELECT COUNT(*) AS count_latest FROM reports_latest")

In [ ]:
# Count the number of reports per sending facility.
# Your row filter (configured by your `allowed_facilities` Keycloak attribute)
# already scopes this to facilities you're entitled to see.
all_facilities = query("""
    SELECT sending_facility, COUNT(*) AS count
    FROM reports_latest
    GROUP BY sending_facility
    ORDER BY sending_facility ASC
""")
display(all_facilities)

# Count reports from IRB-approved institutions; adjust the list to match
# your IRB-approved institutions. Trino will still scope to facilities your
# `allowed_facilities` attribute permits, regardless of what you put here.
FACILITIES = ['BJH', 'WUSM', 'BJCMG', 'SLCH']
FACILITIES_SQL = "'" + "', '".join(FACILITIES) + "'"

query(f"""
    SELECT sending_facility, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY sending_facility
    ORDER BY count DESC
""")

In [ ]:
# Show available modalities and counts, filtering to selected facilities.
query(f"""
    SELECT modality, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY modality
    ORDER BY count DESC
""")

In [ ]:
# Show schema and an example report. SELECT * here to see every column,
# but in practice select only the fields you need. PHI columns
# (`patient_name`, `zip_or_postal_code`, etc.) come back redacted unless
# your `mask_phi_fields` attribute is explicitly set to "false".
sample_row = query(f"""
    SELECT *
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    LIMIT 1
""")
print(sample_row.dtypes)
sample_row.T  # transpose so all columns are visible

In [ ]:
# View sex distribution in CT and MR reports.
query(f"""
    SELECT sex, modality, COUNT(*) AS count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality IN ('CT', 'MR')
    GROUP BY sex, modality
    ORDER BY modality, sex, count DESC
""")

In [ ]:
# Filter reports by date range.
# Relative: last 365 days.
date_filtered = query(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND requested_dt >= CURRENT_DATE - INTERVAL '365' DAY
    LIMIT 10
""")
display(date_filtered)

# Absolute: specific date range.
query(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND requested_dt BETWEEN DATE '2024-01-01' AND DATE '2026-01-01'
    LIMIT 10
""")

In [ ]:
# Search report text with regex.
query(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND regexp_like(report_text, '(?i)pulmonary embolism')
    LIMIT 10
""")

In [ ]:
# Filter by diagnosis code. Trino's `any_match` is the array-predicate
# equivalent of Spark's `exists`.
query(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, diagnoses
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND any_match(diagnoses, d -> d.diagnosis_code IN ('I26.99', 'I26.9'))
    LIMIT 10
""")

In [ ]:
# Combine multiple filters: CT head scans for adults in 2024.
query(f"""
    SELECT accession_number, epic_mrn, patient_age, sex, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality = 'CT'
      AND regexp_like(service_name, '(?i)head|brain')
      AND patient_age >= 18
      AND requested_dt >= DATE '2024-01-01'
    LIMIT 10
""")

In [ ]:
# Show all available patient ID types from the patient_ids array.
# Each report has a `patient_ids` array of structs (id_number,
# assigning_authority, identifier_type_code, assigning_facility). The
# transformer also creates convenience columns like `epic_mrn`, `empi_mr`,
# `bjh_mr`, `slch_mr`, etc. Trino's UNNEST of an array-of-row expands the
# struct's *fields* into columns, so each field gets its own alias.
query(f"""
    SELECT pid_authority, pid_type, COUNT(*) AS count
    FROM reports_latest
    CROSS JOIN UNNEST(patient_ids)
        AS t(pid_id, pid_authority, pid_type, pid_facility)
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY pid_authority, pid_type
    ORDER BY count DESC
""")

In [ ]:
# Random sample of CTs. Trino's `RANDOM()` ordering is the analog of
# Spark's `RAND()`. The query helper already returns a pandas DataFrame —
# no `.toPandas()` needed.
sample = query(f"""
    SELECT accession_number, epic_mrn, birth_date, sex, race, modality, service_name,
           requested_dt, study_instance_uid, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality = 'CT'
    ORDER BY RANDOM()
    LIMIT 100
""")
print(f"Sampled {len(sample)} reports")

In [ ]:
# Inspect the sampled DataFrame.
sample.head()

In [ ]:
# Import a list of patient IDs from a CSV file to look up their reports.
# Upload your own CSV to /home/jovyan/Scout/ via the JupyterHub file browser.
# CSV should have a column named `epic_mrn` — you can also join on any other
# ID column on `reports_latest` (e.g. `study_instance_uid`, `empi_mr`).
patient_ids_path = '/home/jovyan/Scout/patient_ids.csv'
if not os.path.exists(patient_ids_path):
    pd.DataFrame({'epic_mrn': ['REPLACE_WITH_REAL_MRN']}).to_csv(patient_ids_path, index=False)
    print(f"Created sample CSV at {patient_ids_path} — edit it with your own patient IDs")

patient_ids_df = pd.read_csv(patient_ids_path, dtype=str)
mrns = patient_ids_df['epic_mrn'].dropna().str.strip().tolist()
print(f"Loaded {len(mrns)} patient IDs from CSV")

In [ ]:
# Look up reports for the patient list.
mrn_list = "', '".join(mrns)
patient_reports = query(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE epic_mrn IN ('{mrn_list}')
      AND sending_facility IN ({FACILITIES_SQL})
    LIMIT 100
""")
patient_reports.head()

In [ ]:
# Write results to CSV (pandas in-place; no `.write.csv()` needed).
patient_reports.to_csv('/home/jovyan/Scout/patient_list_results.csv', index=False)

## Installing Additional Packages

The base Jupyter environment includes `trino-python-client`, pandas, scipy, matplotlib, seaborn, and other core data-analysis packages. For ML, NLP, or other specialized work, create a new conda environment:

```bash
# Create an environment with specific packages
mamba create -n my-env python=3.11 ipykernel pytorch transformers scikit-learn -y

# Or use the sample environment file (in ~/Scout/environment.yml)
mamba env create -f ~/Scout/environment.yml
```

**Important:** include `ipykernel` in every environment you create. The `nb_conda_kernels` extension uses it to discover the environment as a Jupyter kernel. Without it, your environment won't appear in the kernel list.

After creating an environment, refresh the launcher to see it as a selectable kernel.

Environments live on your persistent home directory (`/home/jovyan/.conda/envs/`) and survive server restarts. In air-gapped deployments, package requests are routed through a proxy transparently — no extra configuration needed.

For more details, see the [Tips & Best Practices](https://washu-scout.readthedocs.io/en/latest/tips.html#installing-additional-packages) documentation.